### Feature Engineering and Data Preprocessing

In [1]:
import pickle
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#### Load Raw Data

In [2]:
dataset = pd.read_csv("../data/churn-modelling.csv")

# Drop irrelevant columns
dataset = dataset.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
dataset.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


#### Feature Engineering

In [3]:
def engineer_features(df):
    """Generates new business-logic features to aid model convergence."""
    df_engineered = df.copy()
    
    # Balance to Salary Ratio
    df_engineered["BalanceSalaryRatio"] = (
        df_engineered["Balance"] / df_engineered["EstimatedSalary"]
    )
    
    # Tenure to Age Ratio
    df_engineered["TenureByAge"] = df_engineered["Tenure"] / df_engineered["Age"]

    # Age Binning
    bins = [18, 30, 40, 50, 60, 100]
    labels = ["Young_Adult", "Adult", "Middle_Age", "Senior", "Elderly"]
    df_engineered["AgeGroup"] = pd.cut(
        df_engineered["Age"], bins=bins, labels=labels, right=False
    )

    # Credit Score Given Age
    df_engineered["CreditScoreGivenAge"] = (
        df_engineered["CreditScore"] / df_engineered["Age"]
    )
    
    # Product holding by Active Status
    df_engineered["Products_Active_Interaction"] = (
        df_engineered["NumOfProducts"] * df_engineered["IsActiveMember"]
    )

    return df_engineered

dataset = engineer_features(dataset)
dataset.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,BalanceSalaryRatio,TenureByAge,AgeGroup,CreditScoreGivenAge,Products_Active_Interaction
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,0.000000,0.047619,Middle_Age,14.738095,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,0.744677,0.024390,Middle_Age,14.829268,1
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1.401375,0.190476,Middle_Age,11.952381,0
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0.000000,0.025641,Adult,17.923077,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,1.587055,0.046512,Middle_Age,19.767442,1


#### Encode Categorical Variables

In [4]:
X = dataset.drop("Exited", axis=1)
y = dataset["Exited"]

In [5]:
# Label Encode Gender
le_gender = LabelEncoder()
X["Gender"] = le_gender.fit_transform(X["Gender"])

In [6]:
# One-Hot Encode Geography and AgeGroup
ct = ColumnTransformer(
    transformers=[
        ("encode", OneHotEncoder(sparse_output=False), ["Geography", "AgeGroup"])
    ],
    remainder="passthrough",
)

In [7]:
cols = ct.fit(X).get_feature_names_out()
X_encoded = pd.DataFrame(
    ct.transform(X), columns=[col.split("__")[-1] for col in cols]
)

X_encoded.head()

,Geography_France,Geography_Germany,Geography_Spain,AgeGroup_Adult,AgeGroup_Elderly,AgeGroup_Middle_Age,AgeGroup_Senior,AgeGroup_Young_Adult,CreditScore,Gender,...,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,BalanceSalaryRatio,TenureByAge,CreditScoreGivenAge,Products_Active_Interaction
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,619.0,0.0,...,2.0,0.00,1.0,1.0,1.0,101348.88,0.000000,0.047619,14.738095,1.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,608.0,0.0,...,1.0,83807.86,1.0,0.0,1.0,112542.58,0.744677,0.024390,14.829268,1.0
2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,502.0,0.0,...,8.0,159660.80,3.0,1.0,0.0,113931.57,1.401375,0.190476,11.952381,0.0
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,699.0,0.0,...,1.0,0.00,2.0,0.0,0.0,93826.63,0.000000,0.025641,17.923077,0.0
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,850.0,0.0,...,2.0,125510.82,1.0,1.0,1.0,79084.10,1.587055,0.046512,19.767442,1.0


#### Train-Validation-Test Split

In [8]:
# Initial split to carve out the 20% Test set
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Secondary split to carve out the 16% Validation set from the remaining training data
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

print(f"Training set size: {X_train.shape}")
print(f"Validation set size: {X_val.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (6400, 21)
Validation set size: (1600, 21)
Test set size: (2000, 21)


#### Feature Scaling

In [9]:
sc = StandardScaler()

# Fit only on training data, then transform all
X_train_scaled = pd.DataFrame(sc.fit_transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(sc.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(sc.transform(X_test), columns=X_test.columns)

#### Export Artifacts and Processed Data

In [10]:
# Save features
X_train_scaled.to_csv("../data/X_train_processed.csv", index=False)
X_val_scaled.to_csv("../data/X_val_processed.csv", index=False)
X_test_scaled.to_csv("../data/X_test_processed.csv", index=False)

# Save targets
y_train.to_csv("../data/y_train.csv", index=False)
y_val.to_csv("../data/y_val.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

with open("../artifacts/label_encoder.pkl", "wb") as f:
    pickle.dump(le_gender, f)
    
with open("../artifacts/onehot_encoder.pkl", "wb") as f:
    pickle.dump(ct, f)
    
with open("../artifacts/standard_scaler.pkl", "wb") as f:
    pickle.dump(sc, f)

print("Artifacts successfully exported!")

Artifacts successfully exported!
